# 例 7.6 · 标普 600 小盘股的一月效应

上一年跌幅最大的股票，次年一月能否跑赢上一年涨幅最大的股票？本例在每年最后一个交易日按全年收益率选股，持有多空组合至次年一月底，计算扣除交易成本后的收益。[MATLAB 原例](../matlab/example7_6.m) · [Python 实现](../python/examples/example7_6.py)

In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy.io import loadmat

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.matlab_compat import matlab_round
from epchan.paths import data_path
from epchan.smartmean import smartmean

ONE_WAY_COST = 0.0005

## 1. 找出年末和次年一月末的交易日

从交易日序列识别 12 月到 1 月、1 月到 2 月的切换点。样本开头的第一个一月没有对应的上年十二月，将它跳过后，每个年末都能与下一年一月末配对。

In [ ]:
data = loadmat(data_path('IJR_20080131.mat'), variable_names=['tday', 'cl'])
dates = data['tday'].ravel().astype(int)
prices = data['cl']
months = dates // 100 % 100
dec_end = np.flatnonzero((months[:-1] == 12) & (months[1:] == 1))
jan_end = np.flatnonzero((months[:-1] == 1) & (months[1:] == 2))[1:]
np.testing.assert_array_equal(dates[dec_end] // 10000 + 1, dates[jan_end] // 10000)

## 2. 配对用于排名的年度收益与持有期收益

第一个年末提供计算全年收益所需的起始价格。从第二个年末起，用相邻两个年末的收盘价计算每只股票的年度收益，再用建仓日和次年一月末的收盘价计算持有期收益。两个矩阵的同一行对应同一次建仓。

In [ ]:
entry = dec_end[1:]
exit_day = jan_end[1:]
prior_prices = prices[dec_end[:-1]]
entry_prices = prices[entry]
annual_returns = (entry_prices - prior_prices) / prior_prices
january_returns = (prices[exit_day] - entry_prices) / entry_prices

## 3. 做多年度输家、做空年度赢家

只用年度收益有效的股票排名。做多收益最低的 10%，做空收益最高的 10%，每组股数按 MATLAB 的规则取整。两边各占组合总名义敞口的一半，组内等权。沿用原例，计算各组一月平均收益时忽略缺失值，开仓和平仓各扣 5 个基点。

In [ ]:
print('年末建仓日   一月平仓日   每组股票数   一月净收益')
for i, annual in enumerate(annual_returns):
    eligible = np.flatnonzero(np.isfinite(annual))
    ranked = eligible[np.argsort(annual[eligible], kind='stable')]
    count = int(matlab_round(len(ranked) / 10))
    losers = smartmean(january_returns[i, ranked[:count]], axis=0)
    winners = smartmean(january_returns[i, ranked[-count:]], axis=0)
    net_return = (losers - winners) / 2 - 2 * ONE_WAY_COST
    print(f'{dates[entry[i]]}     {dates[exit_day[i]]}      {count:3d}       {net_return:.2%}')

### 结果解读

组合在 2006、2007 和 2008 年一月的净收益依次为 **−2.44%、−0.68%、+8.81%**。三个持有期中只有 2008 年一月取得正收益。